# Cross-Hypothesis Summary

Zehner's dissertation centers on one quantity: the dimensionless effective bed conductivity `Lambda/lambda` of a flowed packed bed, modeled (Eq. 66) as a **parallel-resistance structure**

```
Lambda/lambda  ~  lambda_so/lambda  +  Pe / K(d/D)
```

where `lambda_so/lambda` is the analytic stagnant-bed closure (Eqs. 37a/38/42a) and `K(d/D)` is an empirical dispersion coefficient disputed in the literature (Eq. 12 -> K=8, Eq. 70 -> K=9, Agnew-Potter -> opposite d/D trend).

Four independent approaches were run against this structure, each with its own kill criterion:

| # | Notebook | Approach | Kill criterion | Verdict |
|---|---|---|---|---|
| 1 | `Symbolic Regression.ipynb` | PySR: rediscover the additive structure + improve `K(d/D)` | augmented d/D range stays < ~0.05 -> controversy unresolvable | **Partially survives** - structure confirmed, `K~7.2` close to Eq. 12/70; d/D range widened to 0.017-0.19 but discrimination still inconclusive |
| 2 | `ML.ipynb` | GP residual (gray-box) + conformal UQ on Eq. 66 | leave-one-material-out collapses to a constant -> report as oxide-offset, not failure | **In-distribution: strong (-67% RMSE, well-calibrated). Out-of-distribution: fails the calibration kill criterion** |
| 3 | `Multiphysics.ipynb` | Differentiable JAX closure coupled to VDI Fc porous-convection model + Sobol + UQ propagation | n/a (consistency/sensitivity study, no co-located data) | **Closure validated exactly; coupling is physically inactive at lab scale, marginal at industrial scale** |
| 4 | `PINN.ipynb` | Inverse PINN vs. LSQ for `Lambda/lambda` from Abb. 32 profiles | PINN doesn't recover `Lambda/lambda` accurately | **PINN works (0.2-4% error) once correctly configured, but does not beat LSQ on these clean, low-dimensional fits** |

The rest of this notebook walks through each result with its key figure, then draws the cross-cutting conclusions in the final section.

In [ ]:
# === Reproducibility Info ===
import sys
print(f"Python: {sys.version}")

# Package versions
import numpy as np
import pandas as pd
try: import torch; print(f"PyTorch: {torch.__version__}")
except: pass
try: import tensorflow; print(f"TensorFlow: {tensorflow.__version__}")
except: pass
try: import jax; print(f"JAX: {jax.__version__}")
except: pass
try: import sklearn; print(f"scikit-learn: {sklearn.__version__}")
except: pass
try: import scipy; print(f"SciPy: {scipy.__version__}")
except: pass
try: import matplotlib; print(f"Matplotlib: {matplotlib.__version__}")
except: pass

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print("
For full environment, see requirements.txt or environment.yml")
print("For package-specific seeds, see cells below.")


## TL;DR

**Part 1 -- Validation.** Is Zehner's 1972 packed-bed closure `Lambda/lambda = lambda_so/lambda + Pe/K` valid, and where does it break? **It holds.** Four independent methods agree -- and all four degrade at exactly the same edges (T > 500 K, copper, industrial scale, wide d/D), marking genuine physical limits rather than method artifacts.

| Method | Tested | Verdict |
|---|---|---|
| Symbolic Regression | additive structure + K(d/D) | structure confirmed; K ~ 7.2, between Eq. 12 (K=8) and Eq. 70 (K=9) |
| ML (GP + conformal UQ) | scatter reduction + calibration | -67% RMSE in-distribution; honestly fails at T>500K / copper (PICP 47-51% vs 90%) |
| Multiphysics (JAX) | porous-convection coupling + UQ | closure exact (diff 0.0); convection dormant at lab scale, Nu_S ~ 1.09 at industrial scale |
| PINN inverse | parameter recovery vs classical LSQ | works (0.07-9.5% error) but LSQ-shooting is 2000-3700x faster; PINN's value is structural |

**Part 2 -- Application (section 5).** The validated framework, extended with Knudsen pore-gas physics and a calendering-dependent contact term, becomes a quantitative model + QC toolkit for **lithium-ion electrode manufacturing**:

- **Zero-fit predictions** land inside literature ranges across five published datasets (NMC811: **+2.7%**)
- **Calendering-aware closure**: all-27-sheet MAPE **31.1% -> 4.5%** (the data's noise floor); first closure to reproduce the measured u-shape of lambda_eff vs. calendering degree
- **Design principles**: contact parameters decompose into particle plasticity x additive conductivity x binder mechanics -- per-recipe, but rankable and measurable from ~6 calendering states
- **QC tools**: coating porosity to +/-0.008 from a single 3%-noise thermal reading; inline delamination/interface monitoring at z = 5-22

**For practitioners**: Eq. 66 with K=8 is the safe baseline everywhere (+/-10-15%). Add the GP residual only in-distribution (T <= 500 K, known materials). Add convection only for bed depths s > 1 m. For electrodes/separators: `src/electrode_thermal.py` + Electrode.ipynb.

## 1. Symbolic Regression - does the data support `Lambda/lambda = lambda_so/lambda + Pe/K(d/D)`?

**Synthetic recovery (P0)**: PySR, restricted to `{+, -, *, /, log}` and given `(psi, log(kappa))`, recovered low-error closed forms for the analytic `lambda_so/lambda(psi, kappa)` surface - confirming the dimensionless grouping and the pipeline before touching real data.

**Real-data fit, unconstrained (P0)**: run on all 450 measured rows with `(log_kappa, psi, d_over_D, log_Pe, Nu_r)` -> `lambda_ratio_meas`, PySR's Pareto front did **not** spontaneously rediscover the additive `lambda_so/lambda + Pe/K` form - the top picks are dominated by `log_Pe`-only expressions (e.g. `square(square(log_Pe)) * 1.6`, loss 208 at complexity 5). This is expected: across Zehner's narrow `d/D` range, `lambda_so/lambda` varies far less than the `Pe/K` term, so `log_Pe` alone explains most of the variance and SR has no incentive to separate the two terms on its own.

**Constrained variant (P1) - the direct test**: subtracting the *analytic* `lambda_so/lambda` (Eq. 37a/38) and fitting only the residual against `{Pe, d_over_D}` with `{+, -, *, /}`, PySR's best simple model is

```
residual ~ Pe * 0.1384   =>   K = 1 / 0.1384 ~= 7.23
```

**This is the headline result of the SR notebook**: once the stagnant-bed term is subtracted analytically, the residual is essentially `Pe/K` with `K ~ 7.2` - bracketed almost exactly between Eq. 12 (`K=8`) and a bit below Eq. 70 (`K=9`). The parallel-resistance hypothesis is **directly supported** by the data, and the SR-recovered `K` sits closer to the factor-8 literature value than the factor-9 one.

**Per-series K extraction + Abb. 62 augmentation (P1/P2)**: backing out `K_eff = Pe / (Lambda/lambda - lambda_so/lambda)` per series and overlaying the digitized Abb. 62 literature points widens the `d/D` range from Zehner's native `0.017-0.047` to `0.017-0.19` - **above** the 0.05 kill threshold, so the controversy is not fundamentally unresolvable from this combined dataset. The figure below shows the result:

![K(d/D): Zehner data vs. Eq. 12/70 vs. Abb. 62 literature](figures/summary/sr_K_vs_dD.png)

Zehner's own per-series `K_eff` (blue, `d/D` in `[0.017, 0.047]`) sit mostly between the `K=8` and `K=9` lines, scattering `K ~ 4.5-9.5` with no strong trend. The digitized Abb. 62 literature points (orange, `d/D` in `[0.04, 0.19]`) trend upward to `K ~ 9-14.5`, broadly continuing the mild positive `K(d/D)` slope of both Eq. 12 and Eq. 70 (and **not** showing the opposite-sign Agnew-Potter trend).

**Validation (P2)**: extrapolation holdouts on the unconstrained model (#2) give `log-MAE = 0.074` in-distribution vs. `0.140` leave-copper-out (degrades, as expected) but `0.137` leave-largest-`d/D`-out vs. `0.190` in-distribution (the holdout is *easier* than in-distribution here) - a reminder that the unconstrained model's Pareto front is overfit to `log_Pe` and not a reliable extrapolator either way; the constrained-residual result (`K~7.2`) is the more trustworthy finding of this notebook.

**Bottom line**: the additive structure holds up; `K ~ 7-9` is supported across both Zehner's own data and the augmented literature range; the `d/D` controversy is *narrowed* (the data favors the mild-positive-slope camp, factor 8/9, over Agnew-Potter's opposite trend) but not sharply resolved - the widened range (0.017-0.19) is now above the kill threshold, so this is a genuinely informative partial result rather than an unresolvable one.

## 2. ML (gray-box GP residual + conformal UQ) - does it reduce scatter and stay calibrated at the edges?

**Baseline GP, random k-fold (P0)**:

| | RMSE (Lambda/lambda) |
|---|---|
| Eq. 66 baseline | `4.98 +/- 0.46` |
| GP-corrected | `1.66 +/- 0.20` |
| **Improvement** | **66.7%** |

**Conformal UQ (P1)**: at 90% nominal coverage, the split-conformal interval achieves `88.9%` empirical PICP in-distribution, with a median relative interval width of `~10.3%` of `Lambda/lambda` - i.e. roughly the original `+/-10%` scatter, but now as a *calibrated* interval rather than an unquantified bound.

**Extrapolation splits (P1) - the crux**:

| Split | Baseline RMSE | Corrected RMSE | Improvement | PICP @ 90% |
|---|---|---|---|---|
| Random k-fold (in-dist.) | 4.98 | 1.66 | 66.7% | 88.9% |
| Leave-1000K-out | 5.59 | 5.59 | **0.0%** | **50.8%** |
| Leave-copper-out | 6.78 | 5.82 | 14.2% | **46.9%** |
| Pe<1000 -> Pe>=1000 | 9.98 | 2.74 | 72.6% | 100.0% (n=5) |

Two of the three extrapolation splits **fail the calibration kill criterion** outright: leaving out the 1000 K runs gives *zero* improvement (the GP simply can't extrapolate in temperature) and the conformal interval covers only `50.8%` of a nominal-90% target; leaving out copper improves RMSE only modestly (`14.2%` vs. `66.7%` in-distribution) with similarly broken coverage (`46.9%`). Only the `Pe`-extrapolation split looks good, but with only 5 test points it is not statistically meaningful.

**Stratified parity (P2)** - does the correction help only one of Abb. 61's two material groups, or both?

![Stratified parity: Eq. 66 baseline vs. GP-corrected, foam/glass/ceramic/steatite vs. steel/copper](figures/summary/ml_parity_groups.png)

| Group | Baseline RMSE | GP-corrected RMSE |
|---|---|---|
| foam/glass/ceramic/steatite | 4.77 | 1.39 |
| steel/copper | 5.28 | 1.96 |

Both groups tighten substantially and symmetrically (left panel's wide scatter around the dashed parity line collapses to a tight band on the right) - the in-distribution correction is **not** just rebalancing one group at the expense of the other.

**Leave-one-material-out (P2) - what is the GP actually learning?** For each material prefix, train on all others and predict its residual. The **median captured fraction of the held-out material's own mean offset is `-0.03`** - i.e. essentially `0` (the GP predicts close to the global mean, not the held-out material's actual offset; several materials, e.g. `KK10`, even have wildly negative captured fractions). This means the `~67%` in-distribution RMSE reduction is, to first order, **the GP memorizing a per-material constant** - the Eq. 69 oxide-layer offset - rather than learning a continuous function of `(log_kappa, psi, d/D, log_Pe, Nu_r, T, epsilon)`.

**Bottom line**: the GP correction is a real, well-calibrated, ~67% RMSE reduction *in-distribution*, and it helps both of Abb. 61's material groups. But it (a) is largely a per-material lookup rather than new physics (per the kill criterion's own framing, this *is* the reportable finding - it's evidence for the Eq. 69 oxide-offset interpretation), and (b) **fails outright** when pushed to the validity-range edges the project cared about most (high-T, copper extrapolation): RMSE improvement collapses and conformal coverage drops from 90% nominal to ~47-51%. The gray-box model should not be trusted for extrapolation without re-calibration on edge-region data.

## 3. Multiphysics (JAX closure -> VDI Fc porous-convection coupling)

**Differentiable closure (P0)**: `src/zbs_jax.py` reproduces `src/zbs.py` (`lambda_so/lambda`) **exactly** (`max |numpy - jax| = 0.00e+00`) across the full dataset range (`lambda_so/lambda in [2.26, 23.5]`, `psi in [0.369, 0.429]`, `kappa in [4.0, 14094]`) - the closure is now end-to-end differentiable with zero loss of accuracy.

**Fc coupling at lab scale (P1)**: feeding `lambda_Sch <- lambda_so` into the Carman-Kozeny permeability + modified Rayleigh-Darcy number `Ra*_S` gives, at a representative Zehner data point (`psi=0.421, kappa=45.5, d=9.6 mm, s=204 mm`):

```
Ra*_S = 0.96   (Ra*_crit = 39.5)   ->   Nu_S = 1.0  (pure conduction)
```

Sweeping `(psi, kappa)` over the dataset's full range (holding `d, s` fixed at the lab value) confirms this is universal at lab scale:

![Nu_S(psi, lambda_s/lambda) map at lab scale](figures/summary/mp_NuS_map.png)

`Ra*_S` stays in `[0.13, 3.1]` across the entire `(psi, kappa)` grid - **0% of the grid exceeds `Ra*_crit ~= 39.5`**, so `Nu_S = 1` everywhere (rightmost panel is flat). At Zehner's lab scale (`s ~ 0.13-0.35 m`), the bed is firmly **conduction-dominated**: `lambda_so` (not a convective `Nu_S`) is the physically relevant quantity, exactly as the project's framing anticipated.

**Making the sensitivity study non-degenerate (P1)**: sweeping the layer thickness `s` up to industrial-silo scale (`10 m`) crosses the convection threshold: `Ra*_S = 47.0 > Ra*_crit = 39.5` -> `Nu_S = 1.0915`.

**Sensitivities at the industrial-scale point**:

- Local autodiff: `dNu_S/dpsi = 7.46`, `dNu_S/dkappa = -0.0041` - locally, `psi` dominates by ~3 orders of magnitude.
- Global Sobol (over `psi, log_kappa, d, s`): `s` has the largest total effect (`ST=1.46`), `log_kappa` and `d` both have large total effects (`ST~0.83`) but near-zero first-order effects (`S1~0.04, -0.004`) - i.e. their influence is almost entirely through **interactions** (with `s`, near the `Ra*_crit` threshold). `psi`'s total effect is comparatively small (`ST=0.14`) despite its large *local* derivative - the local sensitivity at one point is not representative of the global picture near a threshold/onset phenomenon.

**UQ propagation (P2) - tying Methods 2 and 4 together**: Method 2's GP gives a closure uncertainty of `sigma_r = 0.0204` (~2%, fractional/log-space). Propagated via the delta method through `dNu_S/dlambda_Sch = -1.93` at the industrial-scale point:

```
Nu_S = 1.0915 +/- 0.0111   (90% interval ~ +/- 0.018)
```

**Bottom line**: the closure itself is now an exact, differentiable building block. At Zehner's actual experimental scale, the Fc convective correlation is **inactive** (`Nu_S=1` everywhere) - itself a clean, expected finding. Only by extrapolating to industrial silo scale does the coupling produce a non-trivial (`Nu_S~1.09`) result, right at the onset of convection where global sensitivities are dominated by interaction effects rather than any single parameter - and Method 2's ~2% closure uncertainty propagates to only ~1% uncertainty on `Nu_S`, demonstrating the full closure -> convection -> UQ pipeline end to end even though no co-located data exists to validate it against.

## 4. PINN - inverse recovery of `Lambda/lambda` from Abb. 32 exit-temperature profiles

Working model: `Delta_Theta_2(rho^2) = A * exp(-(Pe / (4 * Lambda/lambda)) * rho^2)`.

![Abb. 32 digitized exit profiles](figures/summary/pinn_abb32_profiles.png)

| Case | LSQ (`curve_fit`) | PINN (per-param-lr Adam + L-BFGS) |
|---|---|---|
| Synthetic, clean (true `Lambda/lambda=666.0`) | `666.0` (0.0%) | `667.2` (0.2%) |
| Synthetic, 5% noise (5 reps) | mean `668.7`, std `14.7` (2.0% err) | mean `687.5`, std `24.5` (3.6% err) |
| Synthetic, partial profile (4/7 pts) | `666.0` (0.0%) | `666.2` (0.0%) |
| Synthetic, multi-run fusion (shared `Lambda`, `Pe=959 & 159`) | `666.0` avg (0.0%) | `663.8` (0.3%) |
| Real Abb. 32, `Pe=959` | `687.9` | `711.4` |
| Real Abb. 32, `Pe=159` | `211.3` | `219.5` |

(SK10 tabulated `lambda_ratio_meas` for comparison: `158.9` at `Pe~959`, `44.4` at `Pe~159` - both LSQ and PINN sit well above these, reflecting the working-model approximation and ~10-15% digitization error, not a fitting-method difference.)

**Why the PINN doesn't beat LSQ here**: the governing equation is a single exponential with two free parameters (`A`, decay rate `b = Pe/(4 Lambda/lambda)`) - exactly what `scipy.optimize.curve_fit` solves in closed-form-friendly fashion. The PINN must additionally fit a ~1700-parameter network to the same shape from the same handful of points (7, or 4 in the partial case), which is a strictly harder optimization for no extra information - hence it is noisier under measurement noise (3.6% vs. 2.0% mean error) and slightly biased on real data, while matching LSQ almost exactly when the fit is well-posed (clean, partial, fused cases: 0.0-0.3% error for both).

**What is genuinely useful**: (1) the multi-run fusion demonstrator shows the PINN can jointly fit a *shared* `Lambda/lambda` across two `Pe` runs in a single training pass, where LSQ needs two independent fits + averaging; (2) the same architecture generalizes directly to non-exponential profiles, the rectangular-source BC (Eq. 51/53), or additional trainable physics parameters - cases where `curve_fit` would need a new closed-form model each time; (3) the practical finding that **scalar inverse-problem variables in DeepXDE need their own (much larger) learning rate** than the network weights - without this, `Lambda` gets stuck near its initial guess regardless of data, even at near-zero training loss.

**Extending to a problem with no closed-form ansatz**: `PINN.ipynb` section 8 replaces the linear exponential with a nonlinear `lambda(Theta)`-coupled dispersion ODE (`dy/drho2 = -(Pe/(4*Lambda_0)) * y/(1+alpha*y)`), which has no closed-form `y(rho2)` -- `curve_fit` can no longer be used directly. Even here, the PINN does not win on accuracy:

| Case | LSQ-shooting (`solve_ivp`+`least_squares`) | PINN |
|---|---|---|
| Synthetic, clean (true `Lambda_0/lambda=666.0`, `alpha=0.0300`) | `666.0` (0.0%), `0.0300` (0.0%) | `663.6` (0.4%), `0.0309` (3.1%) |
| Synthetic, 5% noise (5 reps) | `Lambda_0` mean `648.8` (std `92.9`) | `Lambda_0` mean `574.0` (std `84.3`) |
| Multi-run fusion (shared `Lambda_0, alpha`, clean) | -- | `662.5` (0.5%), `0.0309` (2.9%) |

LSQ-shooting is exact on clean data (it integrates the same ODE used to generate the data) and noticeably more noise-robust than the PINN. A clean-data identifiability sweep confirms `(Lambda_0, alpha)` *are* identifiable from a single 7-point profile, but the cost surface is shallow enough that 5% noise produces large, correlated errors in both parameters for both methods -- a *data* limitation, not a fitting-method one. The PINN's edge remains structural: no bespoke ODE-integration code is needed (the residual is just the ODE written via autodiff), and multi-run fusion across `Pe=959`/`159` reproduces the single-profile accuracy in one training pass.

## 5. Application: battery electrode manufacturing (`Electrode.ipynb`)

**The mapping.** Electrode coatings and separators *are* micrometer-scale packed beds: graphite/NMC particles (d50 ~ 8-17 um) or polymer fibrils (~0.3 um) with air- or electrolyte-filled pores. Their conductivity ratios `kappa = lambda_s/lambda_f` (14-1005) sit **inside** the range validated on Zehner's data [4, 14094] -- the closure *interpolates* in kappa and only *extrapolates* in porosity (electrodes: 0.20-0.55 vs. validated 0.369-0.429; flagged honestly throughout).

**The new physics: Knudsen-extended ZBS.** At these pore sizes the gas mean free path (air 67 nm, He 190 nm) is no longer negligible, and pore-gas conduction is rarefaction-reduced (Smoluchowski):

```
lambda_gas_eff = lambda_gas / (1 + 2*beta*Kn),   Kn = mfp/d_pore,   d_pore = (2/3)*psi/(1-psi)*d_p
```

| System | pore size | Kn (air) | gas-conduction reduction | effect on dry lambda_eff |
|---|---|---|---|---|
| graphite anode (psi=0.30) | 4.9 um | 0.014 | -4.3% | -3.4% |
| NMC cathode (psi=0.30) | 2.3 um | 0.029 | -8.8% | -5.8% |
| **separator (real pores 43-64 nm)** | 43-64 nm | **1.0-1.6** | **-84%** | morphology-dominated (see 5.2) |

![Knudsen reduction of pore-gas conduction](figures/electrode/knudsen_reduction.png)

**Reference values** (W/mK, zero-fit, psi=0.30 electrodes / 0.40 separator): anode wet **2.56** / dry 0.60 -- cathode wet **0.97** / dry 0.31 -- separator wet 0.29 / dry **0.065** (Knudsen) vs 0.116 (continuum). Wetting roughly quadruples anode and triples cathode conductivity, so the *dry* process steps (drying oven, dry room, pre-fill) are the thermally hard ones. A normal +/-0.02 calendering porosity tolerance moves electrode lambda_eff by **5-9%** (exact autodiff sensitivities) -- it is not a material constant.

![lambda_eff vs porosity for anode, cathode, separator](figures/electrode/lambda_eff_vs_porosity.png)

### 5.2 Validation against five published datasets (zero fitting)

All data transcribed with citations into `data/raw/gandert2023_calendering.csv` and `data/raw/separator_electrode_literature.csv`.

![Validation against the Gandert 2023 calendering dataset](figures/electrode/validation_gandert2023.png)

| Test | Result |
|---|---|
| **NMC811** uncalendered (Gandert et al. 2023: 4 electrode types x 27 calendering states, LFA, dry/He) | **+2.7%** -- essentially exact |
| NMC622 / graphite anodes (same source) | -29% / -36 to -39% -- errors ordered *exactly* by binder/contact-bridging strength; the residual-vs-porosity curves define the contact-term calibration target |
| Dry separators, measured 0.07-0.18 W/mK (Richter 2017; Marconnet group 2018: 0.10) | falls between the framework's point-contact lower bound (0.022-0.031) and continuous-skeleton+Knudsen upper bound (0.080-0.155). **Continuum sphere-pack models match separators only by cancelling two errors** (overestimated gas conduction vs. underestimated skeleton connectivity) -- a two-gas-pressure measurement would separate them |
| Soaked/dry ratios (Burheim 2013: 3.0-3.1x; Marconnet 2018: 2.4-2.8x) | model 3.2-4.3x -- slight overestimate = the expected point-contact signature |
| Absolute LCO anchors (dry 0.36 / wet 1.10) | model 0.31 / 0.97 -- within 12-14% |

Notably, Gandert et al. *themselves* flag the Knudsen effect as an unquantified gap in their evaluation -- this framework supplies the missing term (2.8-5.5% in their helium atmosphere), and their model comparison shows both literature models (Oehler, Sangros) failing to reproduce the measured u-shape, which motivates 5.3.

### 5.3 The calendering-aware contact closure, its design principles, and the QC tools

**The closure** (VDI flattened-contact form, `lambda_eff_coating_contact`): bridges between particles carry heat in parallel with the validated point-contact core, with a contact fraction that depends on the compression rate Pi = 1 - s/s_0:

```
k_bed = 1 - sqrt(1-psi) + sqrt(1-psi)*[ phi*kappa_bridge + (1-phi)*lam_pm(psi, kappa) ],   phi(Pi) = max(0, phi0 + a*Pi + b*Pi^2)
```

The quadratic form encodes Gandert's own adhesion observation (shear damage, then interlocking). Calibrated per family on the open data:

| Family | zero-fit MAPE | **calibrated** | fitted (lambda_s, phi0, a, b) |
|---|---|---|---|
| graphite_thin | 27.3% | **1.8%** | (24.8, 0.0094, -0.024, 0) |
| graphite_thick | 48.6% | **5.4%** | (5.0, 0.0121, -0.042, 0.053) |
| NMC622 | 19.1% | **1.4%** | (1.6, 0.0171, -0.089, 0.039) |
| NMC811 | 29.3% | **9.4%** | (1.5, 0.0048, -0.049, 0) |
| **average (27 sheets)** | **31.1%** | **4.5%** | -- at the data's ~5% noise floor |

![Calibrated contact closure vs measured data](figures/electrode/calibrated_contact_closure.png)

**First closure to reproduce the measured u-shape**; phi0 brackets the independent VDI sphere value (0.0077); transfer works within a composition (graphite->graphite 21%) and fails across additive recipes (NMC622->NMC811 40%) -- contact parameters are **per-recipe**.

**Why per-recipe -- the design principles** (volume-space decomposition of the fitted parameters):
1. **Bridge conductance** G = phi0*lambda_b is set by *particle plasticity x additive conductivity x additive volume fraction*: soft graphite flakes self-bridge at **~20x** the effectiveness (G per unit inactive volume: 19-24 vs 1.2-2.3) of additive-bridged hard-particle cathodes; 2 wt% flake graphite additive **doubles** NMC622 over CB-only NMC811.
2. **Shear-damage rate** |a|/phi0 orders perfectly by *binder mechanics*: stiff PVDF loses the network 2-4x faster per unit compression (5.2-10.2 /Pi) than elastomeric CMC/SBR (2.6-3.5 /Pi).
3. **Recovery** b > 0 is a *threshold*, not a constant: present exactly where interlocking is independently evidenced (NMC622's SEM-documented penetration into the soft Al foil; the highest-line-load thick anode). NMC811's b=0 = regime not reached (Pi <= 0.26).

Design rules: prefer a small flake-graphite additive over more carbon black (~2x); use elastomeric co-binder for heavily calendered designs; don't park the calendering setpoint in the conductivity/adhesion dip -- calender lighter or push through to interlocking.

**The QC tools** (JAX-differentiable, Newton inversion exact to 1e-16):
- **Porosity QC**: a single 3%-noise thermal reading resolves coating porosity to **+/-0.008 absolute** (200-sheet demonstrator: RMSE 0.0078, predicted sigma 0.0092; 87.5% single-shot spec classification at +/-0.02, cleanly resolvable with n=4 averaging).
- **Interface/delamination QC**: the calibrated closure decomposes lambda_eff into core + bridge shares -- as-coated, bridges carry **16-66%** of the heat, so bridge-network failure shifts a thermal reading by **z = 5-22** sigma at 3% noise. Porosity is already measurable inline; *interface quality is not* -- this is its most direct inline proxy.

![Inverse porosity QC on a 200-sheet batch](figures/electrode/inverse_porosity_qc.png)

**Honest limits**: psi extrapolated below 0.369; lambda_s/phi0 partially confounded at Pi=0 (grouped quantities are the trustworthy ones); stack-derived literature values include interlayer contact resistance; wet separator kappa=2.2 below the validated minimum. **Recalibration path**: ~20-30 sheets at known porosity (dry + soaked, two gas pressures to separate Knudsen from skeleton conduction) pins C, phi and the GP residual on in-house data -- every tool for that campaign exists in this repo.

**Publication package**: dataset provenance, exact architectures/hyperparameters, the quantified error analysis (transfer decomposition + identifiability, Electrode.ipynb 7.3 + `figures/electrode/error_analysis.png`), and the in-house lab protocol are consolidated in `PUBLICATION_METHODS.md`.

## 6. Cross-cutting synthesis

**The existing closure holds up.** All four independent methods are consistent with - and none overturns - Zehner's own structure `Lambda/lambda ~= lambda_so/lambda + Pe/K`:

- SR's constrained residual fit gives `K ~ 7.2`, between Eq. 12 (`K=8`) and Eq. 70 (`K=9`), and the augmented `K(d/D)` trend (Zehner + Abb. 62) is mildly positive - consistent with both Eq. 12/70 and **inconsistent** with Agnew-Potter's opposite-sign claim.
- Multiphysics validates `lambda_so/lambda(psi, kappa)` to floating-point precision in a differentiable form.
- ML's GP residual on top of Eq. 66 cuts in-distribution RMSE by `66.7%`, with both Abb. 61 material groups improving together.
- PINN's inverse fits agree with LSQ to within `0.2-4%` on the linear (closed-form) profile, and even on a nonlinear `lambda(Theta)`-coupled ODE with no closed-form solution, LSQ-shooting (`solve_ivp`+`least_squares`) remains exact on clean data and more noise-robust than the PINN.

**Every method degrades at exactly the "validity-range edges" the project was designed to probe.** This is the most consistent cross-cutting finding:

| Method | Edge condition | Result |
|---|---|---|
| SR | augmented `d/D` range (0.017-0.19) | controversy narrowed, not resolved |
| ML | `T=1000K` / copper extrapolation | RMSE improvement -> `0%`/`14%`, PICP -> `51%`/`47%` (vs. 90% nominal) |
| Multiphysics | lab scale -> industrial scale (`s: 0.2m -> 10m`) | `Nu_S` goes from exactly `1` (inactive) to barely `1.09` (onset) |
| PINN | noisy / real data, and noisy nonlinear (no-closed-form) case | error grows from `0.2%` to `3.6%` (noise) / `~4%` (real data); on the nonlinear case, 5% noise pushes `Lambda_0` error to ~14% for both methods (PINN slightly worse) |

**What each notebook actually delivers, beyond a single number:**

1. **Symbolic Regression** - a falsifiable, interpretable confirmation of the parallel-resistance structure and a `K` estimate that discriminates between competing literature claims (favoring Eq. 12/70 over Agnew-Potter).
2. **ML** - quantitative evidence that the `+/-10%` scatter in Abb. 61 is largely a *per-material* (oxide-layer) effect rather than a smooth function of the dimensionless groups, plus a calibrated UQ pipeline that **correctly flags its own untrustworthiness** at the validity-range edges (this is itself the kill-criterion-anticipated outcome).
3. **Multiphysics** - a reusable, exact, differentiable closure (`src/zbs_jax.py`) plus a full closure -> permeability -> Rayleigh-Darcy -> Nusselt -> UQ pipeline, with the honest finding that the convective coupling is dormant at the scale Zehner actually measured.
4. **PINN** - a working, generalizable inverse-PDE pipeline (with a documented, reusable DeepXDE training recipe) that matches classical LSQ on simple exponential profiles and remains competitive (though not superior) on a nonlinear, no-closed-form ODE where LSQ instead falls back to `solve_ivp`-based shooting. Its lasting value is structural: one-line extension to extra trainable physics parameters and single-pass multi-run fusion, both of which would require new bespoke code for LSQ.

**Overall**: none of the four approaches found a *better* predictive model than Eq. 66 + Eq. 37a/38 with `K~8-9` for the conditions Zehner actually measured - and that is itself a meaningful, well-supported result. Their combined value lies in (a) sharpening *which* parts of the existing model are well-supported (the additive structure, `K~7-9`) vs. under-determined (the precise `d/D` trend, the physical origin of the residual scatter), (b) precisely characterizing *where* the model's reliability breaks down (T/material extrapolation, industrial scale-up, noisy inverse problems), and (c) leaving behind reusable, differentiable/inverse-capable tooling for any future work where additional data (wider `d/D`, higher `T`, co-located convection measurements, more profile points) becomes available.

**The application test (section 5)**: the framework's reuse value was demonstrated end-to-end on a different industry. The differentiable closure (Method 3) became the forward model; Method 2's residual scatter became the explicit model-form uncertainty floor; Method 4's inverse lessons became the porosity and interface QC; and Method 1's core lesson -- *fit only what the known physics cannot explain* -- became the contact-term calibration strategy that took the electrode benchmark from 31.1% to 4.5% MAPE. One framework, validated once, redeployed with quantified honesty.